# 📊 Crypto Analyzer in Google Colab

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bademeischta/crypto_analyzer/blob/v2/notebooks/crypto_analyzer_colab.ipynb)

KI-gestützte Krypto-Analyse mit **ehrlicher Out-of-Sample-Bewertung** – direkt im Browser, ohne Installation.

> ⚠️ **Nur zu Bildungszwecken – keine Finanzberatung.** Krypto-Märkte sind hochspekulativ. Handle niemals auf Basis
> von KI-Vorhersagen allein. Vergangene Performance – auch im Backtest – garantiert keine zukünftigen Ergebnisse.

**So geht's:** Zellen von oben nach unten ausführen (▶ bzw. `Umschalt + Enter`). Die Felder rechts in den Zellen
(Coin, Intervall, …) lassen sich direkt ändern – danach die Zelle erneut ausführen.

| Abschnitt | Inhalt |
|---|---|
| **1 · Setup** | Code laden & Pakete installieren (beim ersten Mal ca. 1 Minute) |
| **2 · Analyse** | Signal, Wahrscheinlichkeiten, Modellqualität und Charts für einen Coin |
| **3 · Backtest** | Signalstrategie vs. Buy & Hold mit eigenen Parametern |
| **4 · Vergleich** | Mehrere Coins: Performance, Risiko, Korrelation |
| **5 · Scanner** | Die liquidesten Coins auf einen Blick |
| **6 · Dashboard** | Die komplette Streamlit-Oberfläche über einen öffentlichen Link |

In [ ]:
#@title 1 · Setup: Code laden & Pakete installieren { display-mode: "form" }
#@markdown Lädt das Projekt von GitHub und installiert fehlende Pakete.
BRANCH = "v2"  #@param {type:"string"}
#@markdown Cache und trainierte Modelle in Google Drive speichern (überleben einen Neustart der Laufzeit):
IN_DRIVE_SPEICHERN = False  #@param {type:"boolean"}

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Bademeischta/crypto_analyzer.git"
IN_COLAB = "google.colab" in sys.modules


def sh(cmd: str, check: bool = True) -> str:
    """Shell-Befehl ausführen; Ausgabe zurückgeben, bei Fehler abbrechen."""
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    res = subprocess.run(cmd, shell=True, capture_output=True, encoding="utf-8", errors="replace", env=env)
    if check and res.returncode != 0:
        raise RuntimeError(f"Befehl fehlgeschlagen: {cmd}\n{res.stdout}\n{res.stderr}")
    return res.stdout + res.stderr


if IN_COLAB:
    PROJECT = Path("/content/crypto_analyzer")
    if not PROJECT.exists():
        try:
            sh(f"git clone -q --depth 1 -b {BRANCH} {REPO_URL} {PROJECT}")
        except RuntimeError:
            print(f"Branch '{BRANCH}' nicht gefunden – nutze den Standard-Branch.")
            sh(f"git clone -q --depth 1 {REPO_URL} {PROJECT}")
    else:
        sh(f"git -C {PROJECT} pull -q", check=False)
    os.chdir(PROJECT)
    print("Installiere Pakete …")
    sh(f"{sys.executable} -m pip install -q -r requirements.txt")

    if IN_DRIVE_SPEICHERN:
        from google.colab import drive

        drive.mount("/content/drive")
        target = Path("/content/drive/MyDrive/crypto_analyzer_data")
        target.mkdir(parents=True, exist_ok=True)
        data = PROJECT / "data"
        if data.exists() and not data.is_symlink():
            sh(f"cp -rn '{data}/.' '{target}/' && rm -rf '{data}'")
        if not data.exists():
            data.symlink_to(target, target_is_directory=True)
        print(f"Daten werden in Google Drive gespeichert: {target}")
else:
    # Lokal in Jupyter: Das Notebook liegt in notebooks/ → Projektwurzel eine Ebene höher
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(PROJECT)

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

branch = sh("git rev-parse --abbrev-ref HEAD", check=False).strip()
print(f"✅ Bereit: {PROJECT} · Branch {branch} · Python {sys.version.split()[0]}")

In [ ]:
#@title Systemcheck: Pakete & API-Erreichbarkeit { display-mode: "form" }
#@markdown Colab-Server stehen meist in den USA, wo `api.binance.com` gesperrt ist (HTTP 451).
#@markdown Das ist kein Problem, solange der **Binance Mirror** erreichbar ist – die App wechselt automatisch.
print(sh(f"{sys.executable} main.py check", check=False))

## 2 · Analyse eines Coins

Lädt Kursdaten, berechnet Indikatoren, trainiert das Modell per *purged Walk-Forward* (nur beim ersten Mal bzw. nach
Ablauf, danach aus dem Cache) und bewertet es auf Daten, die es beim Training nie gesehen hat.

**Lesehilfe:** *Skill* > 0 heißt „weiß mehr als Raten nach Klassenhäufigkeit", *p* < 0,05 heißt „statistisch
signifikant". Für die **Kursrichtung** ist das bei keinem getesteten Top-Coin der Fall – das Modell zeigt dann ehrlich
„kein klares Signal". Die **Volatilität** ist dagegen oft signifikant vorhersagbar.

In [ ]:
#@title 2 · Coin analysieren { display-mode: "form" }
SYMBOL = "BTC"  #@param {type:"string"}
INTERVALL = "4h"  #@param ["1h", "4h", "1d"]
CHART_TAGE = 180  #@param {type:"slider", min:30, max:1460, step:30}
NEU_TRAINIEREN = False  #@param {type:"boolean"}

import logging
import warnings

import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)
pd.set_option("display.width", 200)

from src.analysis.analyzer import CryptoAnalyzer
from src.analysis.report import format_summary
from src.ui import components as ui

if "analyzer" not in globals():
    analyzer = CryptoAnalyzer()
SCHWELLE_KONFIDENZ = analyzer.config["ml"]["confidence_display_threshold"]

result = analyzer.analyze(SYMBOL, INTERVALL, CHART_TAGE, force_retrain=NEU_TRAINIEREN,
                          progress=lambda msg: print("…", msg))
print()
print(format_summary(result))

if not result.error:
    signale = ui.historical_signals(result.oof, SCHWELLE_KONFIDENZ)
    ui.price_chart(result.ohlcv, result.symbol, analyzer.config["features"], signale).show()
    ui.indicator_panel(result.ohlcv, analyzer.config["features"]).show()
    display(ui.indicator_summary(result.ohlcv, analyzer.config["features"]))

In [ ]:
#@title Modellqualität im Detail { display-mode: "form" }
m = result.eval_metrics
if m is None:
    print("Keine Modell-Evaluierung verfügbar:", result.ml_error or result.error)
else:
    print(m.disclaimer, "\n")
    print("Log-Loss je Testperiode (Modell vs. Raten nach Klassenhäufigkeit, niedriger = besser):")
    ui.fold_chart(m).show()
    print("Kalibrierung – stimmen die Wahrscheinlichkeiten?")
    ui.calibration_chart(m.calibration).show()
    ui.confusion_chart(m.confusion, ["BEARISH", "NEUTRAL", "BULLISH"]).show()
    print("Was geschah nach Signalen über der Konfidenzschwelle?")
    display(ui.signal_stats_table(m.signal_stats, m.unconditional_avg_return))
    if result.feature_importance:
        print("Feature-Wichtigkeit (Permutation, out-of-sample):")
        ui.feature_importance_chart(result.feature_importance).show()

## 3 · Backtest

Simuliert die Modellsignale auf den **Out-of-Sample-Vorhersagen** des Walk-Forward: Jede Entscheidung basiert nur
auf Daten, die zu diesem Zeitpunkt bekannt waren. Gibt es bei der gewählten Schwelle keine Signale, war das Modell
nie so sicher – dann die Schwelle senken.

In [ ]:
#@title 3 · Backtest { display-mode: "form" }
KONFIDENZSCHWELLE = 0.5  #@param {type:"slider", min:0.34, max:0.8, step:0.01}
KOSTEN_PRO_SEITE_BPS = 15  #@param {type:"number"}
MINDESTHALTEDAUER_KERZEN = 1  #@param {type:"integer"}
SHORT_ERLAUBEN = False  #@param {type:"boolean"}

from src.models.backtest import run_backtest

if result.oof is None:
    print("Kein Backtest möglich:", result.ml_error or result.error)
else:
    bt = run_backtest(result.oof, KONFIDENZSCHWELLE, result.periods_per_year, fee_bps=KOSTEN_PRO_SEITE_BPS,
                      slippage_bps=0, allow_short=SHORT_ERLAUBEN, min_hold_bars=MINDESTHALTEDAUER_KERZEN)
    print(f"Zeitraum: {bt.params['start'][:10]} → {bt.params['end'][:10]}")
    if bt.strategy["n_trades"] == 0:
        print(f"Bei Schwelle {KONFIDENZSCHWELLE:.0%} gab es kein einziges Signal – Schwelle senken.")
    display(ui.backtest_table(bt))
    ui.equity_chart(bt).show()

## 4 · Coin-Vergleich

Relative Performance (Start = 100), Risiko-Kennzahlen und Korrelation der Returns. Werte nahe 1 in der
Korrelationsmatrix bedeuten: Die Coins laufen fast im Gleichschritt – wenig Diversifikation.

In [ ]:
#@title 4 · Coins vergleichen { display-mode: "form" }
COINS = "BTC, ETH, SOL, DOGE"  #@param {type:"string"}
VERGLEICH_TAGE = 180  #@param {type:"slider", min:30, max:730, step:30}

cmp = analyzer.compare([s for s in COINS.replace(";", ",").split(",") if s.strip()], INTERVALL, VERGLEICH_TAGE)
for sym, err in cmp.errors.items():
    print(f"⚠️ {sym}: {err}")
if not cmp.normalized.empty:
    ui.comparison_chart(cmp.normalized).show()
    tabelle = cmp.stats.copy()
    for col in ("Return", "Volatilität (ann.)", "Max. Drawdown", "Abstand zum Hoch"):
        tabelle[col] = tabelle[col].map(lambda v: f"{v:+.1%}")
    for col in [c for c in tabelle.columns if c.startswith(("Sharpe", "Beta"))]:
        tabelle[col] = tabelle[col].map(lambda v: f"{v:.2f}")
    display(tabelle)
    ui.correlation_heatmap(cmp.correlation).show()

## 5 · Scanner

Analysiert die liquidesten Coins (nach Binance-Volumen) und zeigt Signale **zusammen mit der gemessenen
Modellqualität**. Der erste Lauf trainiert für jeden Coin ein Modell (in Colab ca. 5–15 s pro Coin).

In [ ]:
#@title 5 · Top-Coins scannen { display-mode: "form" }
ANZAHL_COINS = 8  #@param {type:"slider", min:3, max:30, step:1}

symbole = [row["symbol"] for row in analyzer.get_universe(ANZAHL_COINS)] or ["BTC", "ETH", "SOL"]
scan = analyzer.scan(symbole, INTERVALL, progress=lambda done, total, sym: print(f"{done}/{total} {sym}"),
                     max_workers=2)
with pd.option_context("display.float_format", "{:.3f}".format, "display.max_columns", 30):
    display(scan.drop(columns=["Hinweis"], errors="ignore"))

## 6 · Dashboard (komplette Oberfläche)

Startet die Streamlit-App im Hintergrund und erzeugt über einen **Cloudflare-Tunnel** einen öffentlichen Link
(`…trycloudflare.com`, kein Account nötig). Der Link funktioniert, solange diese Colab-Sitzung läuft.

> 🔒 Wer den Link kennt, kann die App öffnen. Teile ihn nur, wenn du das möchtest.

In [ ]:
#@title 6 · Dashboard starten { display-mode: "form" }
import re
import socket
import time
import urllib.request


def port_offen(port: int) -> bool:
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0


if not IN_COLAB:
    print("Lokal startest du das Dashboard einfach im Terminal:  python main.py")
else:
    if not port_offen(8501):
        subprocess.Popen(
            [sys.executable, "-m", "streamlit", "run", "main.py", "--server.headless", "true", "--server.port", "8501"],
            stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT, start_new_session=True,
        )
        for _ in range(60):
            if port_offen(8501):
                break
            time.sleep(1)

    cloudflared = Path("/content/cloudflared")
    if not cloudflared.exists():
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared
        )
        cloudflared.chmod(0o755)

    if not globals().get("DASHBOARD_URL"):
        tunnel_log = Path("/content/cloudflared.log")
        subprocess.Popen(
            [str(cloudflared), "tunnel", "--no-autoupdate", "--url", "http://localhost:8501"],
            stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT, start_new_session=True,
        )
        DASHBOARD_URL = None
        for _ in range(60):
            time.sleep(1)
            treffer = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_log.read_text(errors="ignore"))
            if treffer:
                DASHBOARD_URL = treffer.group(0)
                break

    if DASHBOARD_URL:
        from IPython.display import HTML

        display(HTML(f'<a href="{DASHBOARD_URL}" target="_blank" style="font-size:1.3em">🚀 Dashboard öffnen: '
                     f'{DASHBOARD_URL}</a>'))
        print("Der erste Aufruf eines Coins dauert ein paar Sekunden (Modelltraining).")
    else:
        print("Tunnel-URL nicht gefunden – Details in /content/cloudflared.log und /content/streamlit.log")

## Hinweise

- **Nichts bleibt dauerhaft:** Nach einem Neustart der Laufzeit sind Cache und Modelle weg (außer mit
  *In Drive speichern* in Schritt 1). Neutraining kostet nur wenige Sekunden pro Coin.
- **Binance aus den USA:** Colab-Server stehen meist in den USA; die App nutzt dann automatisch den offiziellen
  Market-Data-Mirror `data-api.binance.vision`.
- **CoinGecko** teilt sich das kostenlose Rate-Limit mit allen Colab-Nutzern derselben IP – Marktdaten (Market Cap,
  Rang) können daher fehlen oder etwas älter sein. Kurse, Modell und Signale betrifft das nicht.
- **Neue Version holen:** Schritt 1 erneut ausführen (macht ein `git pull`).
- Code, Tests und Dokumentation: [github.com/Bademeischta/crypto_analyzer](https://github.com/Bademeischta/crypto_analyzer)